# 🏥 ND Dataset Preprocessing & Multi-Dataset Integration

## Overview
This notebook performs **Stage 1 bias-mitigation preprocessing** on the **ND (New Dataset)** folder and **integrates** all datasets from both `Data_Set/` and `ND/` folders.

| Folder | Datasets | Status |
|--------|----------|--------|
| `Data_Set/` | UCI Heart Disease, CKD, eICU (eLCU), NHANES, CDC Diabetes, Breast Cancer Coimbra | ✅ Already preprocessed |
| `ND/` | MIMIC-IV v3.1, eICU-CRD v2.0, PTB-XL ECG, HAM10000, Heart Failure EHR, MIMIC-CXR | 🔄 Preprocessing in this notebook |

## Pipeline Stages
1. **Environment Setup** — Libraries, paths, constants
2. **Shared Algorithms** — 7 bias-mitigation algorithms (C-Chart DI, MICE, Reweighing, DI Remover, GAN Aug, ProWSyn, Counterfactual)
3. **ND Preprocessing** — Load, clean, apply 7 algorithms, fairness audit (before/after)
4. **Load Data_Set Outputs** — Load already-preprocessed CSVs
5. **Cross-Dataset Integration** — Unified schema, domain grouping, full merge
6. **Quality Validation** — Summary tables, visualizations, bias dashboard
7. **Export** — Save all outputs

---
## 🔧 Stage 1: Environment Setup

In [ ]:
# ── Install required packages (run once) ──
import subprocess, sys
required = ['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'scipy', 'tqdm', 'openpyxl']
for pkg in required:
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('✅ All packages ready')

In [ ]:
import os, warnings, ast, json, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from typing import Optional, Dict, List, Tuple
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.neighbors import NearestNeighbors
from collections import Counter
from scipy import stats
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 30)
pd.set_option('display.float_format', '{:.3f}'.format)

# ── Plotting style ──
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('ggplot')
sns.set_palette('husl')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# ── Path Configuration ──
BASE    = Path(r'e:/Data_Set')
DS_DIR  = BASE / 'Data_Set'
ND_DIR  = BASE / 'ND'
DS_PP   = DS_DIR / 'preprocessed_output'       # Already preprocessed Data_Set outputs
OUT_DIR = BASE / 'integrated_output'
ND_PP   = OUT_DIR / 'nd_preprocessed'           # ND preprocessed outputs

OUT_DIR.mkdir(exist_ok=True)
ND_PP.mkdir(exist_ok=True)
(OUT_DIR / 'by_domain').mkdir(exist_ok=True)
(OUT_DIR / 'plots').mkdir(exist_ok=True)

print(f'BASE    : {BASE}  | exists: {BASE.exists()}')
print(f'DS_DIR  : {DS_DIR}  | exists: {DS_DIR.exists()}')
print(f'ND_DIR  : {ND_DIR}  | exists: {ND_DIR.exists()}')
print(f'DS_PP   : {DS_PP}  | exists: {DS_PP.exists()}')
print(f'OUT_DIR : {OUT_DIR}')
print(f'ND_PP   : {ND_PP}')

# ── ND Dataset Path Registry ──
ND_PATHS = {
    'mimic_patients'   : ND_DIR / 'mimic-iv-3.1' / 'hosp' / 'patients.csv.gz',
    'mimic_admissions' : ND_DIR / 'mimic-iv-3.1' / 'hosp' / 'admissions.csv.gz',
    'mimic_diagnoses'  : ND_DIR / 'mimic-iv-3.1' / 'hosp' / 'diagnoses_icd.csv.gz',
    'mimic_icustays'   : ND_DIR / 'mimic-iv-3.1' / 'icu'  / 'icustays.csv.gz',
    'eicu2_patient'    : ND_DIR / 'eicu-collaborative-research-database-2.0' / 'patient.csv.gz',
    'eicu2_diagnosis'  : ND_DIR / 'eicu-collaborative-research-database-2.0' / 'diagnosis.csv.gz',
    'ptbxl_db'         : ND_DIR / 'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'
                                / 'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'
                                / 'ptbxl_database.csv',
    'ptbxl_scp'        : ND_DIR / 'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'
                                / 'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'
                                / 'scp_statements.csv',
    'ptbxl_features'   : ND_DIR / 'ptb-xl-a-comprehensive-electrocardiographic-feature-dataset-1.0.1'
                                / 'ptb-xl-a-comprehensive-electrocardiographic-feature-dataset-1.0.1'
                                / 'features' / '12sl_features.csv',
    'ham_metadata'     : ND_DIR / 'dataverse_files' / 'HAM10000_metadata',
    'ham_gt'           : ND_DIR / 'dataverse_files' / 'ISIC2018_Task3_Test_GroundTruth.csv',
    'hf_dat'           : ND_DIR / 'hospitalized-patients-with-heart-failure-integrating-electronic-healthcare-records-and-external-outcome-data-1.2'
                                / 'hospitalized-patients-with-heart-failure-integrating-electronic-healthcare-records-and-external-outcome-data-1.2'
                                / 'dat.csv',
    'hf_dat_md'        : ND_DIR / 'hospitalized-patients-with-heart-failure-integrating-electronic-healthcare-records-and-external-outcome-data-1.2'
                                / 'hospitalized-patients-with-heart-failure-integrating-electronic-healthcare-records-and-external-outcome-data-1.2'
                                / 'dat_md.csv',
    'mimic_cxr_dir'    : ND_DIR / 'mimic-cxr-reports' / 'files',
}

# ── Verify ND files exist ──
print(f'\n{"Dataset":<30} {"Exists":<8} {"Size"}')
print('-' * 55)
for name, path in ND_PATHS.items():
    exists = path.exists()
    if exists and path.is_file():
        size = f'{path.stat().st_size / 1e6:.2f} MB'
    elif exists and path.is_dir():
        size = 'DIR'
    else:
        size = 'N/A'
    status = '✅' if exists else '❌'
    print(f'{status} {name:<30} {size}')

print('\n✅ Environment setup complete')

---
## ⚙️ Stage 2: Shared Algorithms & Utilities

### 2.1 Fairness Metrics

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FAIRNESS METRICS (shared across all datasets)
# ═══════════════════════════════════════════════════════════════════════════

def compute_spd(df, target_col, sensitive_col, priv_val, unpriv_val):
    """Statistical Parity Difference."""
    priv   = df[df[sensitive_col] == priv_val]
    unpriv = df[df[sensitive_col] == unpriv_val]
    if len(priv) == 0 or len(unpriv) == 0:
        return float('nan')
    return unpriv[target_col].mean() - priv[target_col].mean()


def compute_di(df, target_col, sensitive_col, priv_val, unpriv_val):
    """Disparate Impact."""
    priv   = df[df[sensitive_col] == priv_val]
    unpriv = df[df[sensitive_col] == unpriv_val]
    if len(priv) == 0 or len(unpriv) == 0:
        return float('nan')
    p_priv   = priv[target_col].mean()
    p_unpriv = unpriv[target_col].mean()
    if p_priv == 0:
        return float('nan')
    return p_unpriv / p_priv


def fairness_audit(df, target_col, sensitive_col, priv_val, unpriv_val, label=''):
    """Compute and print SPD and DI."""
    spd = compute_spd(df, target_col, sensitive_col, priv_val, unpriv_val)
    di  = compute_di(df, target_col, sensitive_col, priv_val, unpriv_val)
    prevalence = df[target_col].mean() * 100

    print(f'    {label}')
    print(f'      Prevalence: {prevalence:.1f}%')
    print(f'      SPD: {spd:+.4f}' if not np.isnan(spd) else '      SPD: N/A')
    print(f'      DI:  {di:.4f}'  if not np.isnan(di)  else '      DI:  N/A')

    if not np.isnan(spd) and abs(spd) > 0.10:
        print(f'      [!] SPD exceeds +/-0.10 threshold')
    if not np.isnan(di) and (di < 0.80 or di > 1.25):
        print(f'      [!] DI violates 80% rule (0.80-1.25)')

    return {
        'SPD':        round(spd, 4) if not np.isnan(spd) else 'N/A',
        'DI':         round(di, 4)  if not np.isnan(di)  else 'N/A',
        'Prevalence': round(prevalence, 1)
    }

print('✅ Fairness metrics defined')

### 2.2 Seven Stage-1 Pre-Processing Algorithms

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 7 PRE-PROCESSING ALGORITHMS (reusable for all datasets)
# Ported from: Data_Set/National Center for health Statistics/multi_dataset_preprocessing.py
# ═══════════════════════════════════════════════════════════════════════════

def algo1_cchart_di_imputation(df, sensitive_col, target_col):
    """Algorithm 1: C-Chart Based DI Imputation Selection."""
    print('    [Algo 1] C-Chart Based DI Imputation Selection')
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    num_cols = [c for c in num_cols if c != target_col and c != sensitive_col]

    cols_missing = [c for c in num_cols if df[c].isnull().sum() > 0]
    if not cols_missing:
        print('      No missing values -- skipped')
        return df

    strategies = ['mean', 'median', 'most_frequent']
    methods = {}
    for col in cols_missing:
        if df[col].dropna().empty:
            df[col] = 0
            methods[col] = 'zero_fill'
            continue
        best_di, best_s = float('inf'), 'mean'
        for s in strategies:
            try:
                imp  = SimpleImputer(strategy=s)
                vals = imp.fit_transform(df[[col]]).ravel()
                tmp  = df.copy()
                tmp[col] = vals
                med = tmp[col].median()
                if pd.isna(med):
                    continue
                tmp['_p'] = (tmp[col] > med).astype(int)
                di = compute_di(tmp, '_p', sensitive_col, 1, 0)
                if not np.isnan(di) and abs(di - 1.0) < best_di:
                    best_di = abs(di - 1.0)
                    best_s  = s
            except Exception:
                continue
        methods[col] = best_s
        try:
            imp = SimpleImputer(strategy=best_s)
            df[col] = imp.fit_transform(df[[col]]).ravel()
        except Exception:
            df[col] = df[col].fillna(df[col].mean())

    counts = Counter(methods.values())
    print(f'      Methods: {dict(counts)}, Columns: {len(methods)}')
    return df


def algo2_mice(df, target_col):
    """Algorithm 2: MICE (Multiple Imputation by Chained Equations)."""
    print('    [Algo 2] MICE (Multiple Imputation by Chained Equations)')
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    num_cols = [c for c in num_cols if c != target_col]
    missing  = df[num_cols].isnull().sum().sum()
    if missing == 0:
        print('      No missing values -- skipped')
        return df
    mice = IterativeImputer(max_iter=10, random_state=42, sample_posterior=False)
    df[num_cols] = mice.fit_transform(df[num_cols])
    print(f'      Missing: {missing} -> {df[num_cols].isnull().sum().sum()}')
    return df


def algo3_reweighing(df, sensitive_col, target_col):
    """Algorithm 3: Reweighing."""
    print(f'    [Algo 3] Reweighing (target: {target_col})')
    n = len(df)
    weights = np.ones(n)
    for g in df[sensitive_col].unique():
        for y in df[target_col].unique():
            p_g  = (df[sensitive_col] == g).sum() / n
            p_y  = (df[target_col] == y).sum() / n
            mask = (df[sensitive_col] == g) & (df[target_col] == y)
            p_obs = mask.sum() / n
            if p_obs > 0:
                weights[mask] = (p_g * p_y) / p_obs
    df['Sample_Weight'] = weights
    print(f'      Weights: [{weights.min():.4f}, {weights.max():.4f}], Mean: {weights.mean():.4f}')
    return df


def algo4_di_remover(df, sensitive_col, target_col, repair=0.8):
    """Algorithm 4: Disparate Impact Remover."""
    print(f'    [Algo 4] Disparate Impact Remover (repair={repair})')
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    num_cols = [c for c in num_cols if c != target_col and c != sensitive_col and c != 'Sample_Weight']
    groups = df[sensitive_col].dropna().unique()
    count  = 0
    for col in num_cols:
        if df[col].isnull().all():
            continue
        medians = {}
        for g in groups:
            d = df.loc[df[sensitive_col] == g, col].dropna()
            if len(d) > 0:
                medians[g] = d.median()
        if len(medians) < 2:
            continue
        global_med = df[col].median()
        df[col] = df[col].astype('float64')
        for g in groups:
            if g in medians:
                shift = float((global_med - medians[g]) * repair)
                mask  = df[sensitive_col] == g
                df.loc[mask, col] = df.loc[mask, col].values + shift
        count += 1
    print(f'      Features repaired: {count}')
    return df


def algo5_gan_augmentation(df, sensitive_col, target_col):
    """Algorithm 5: GAN-Based Data Augmentation."""
    print(f'    [Algo 5] GAN-Based Data Augmentation')
    groups = df[sensitive_col].dropna().unique()
    gc_tab = df.groupby(sensitive_col)[target_col].value_counts().unstack(fill_value=0)
    max_c  = gc_tab.values.max()
    num_cols = [c for c in df.select_dtypes(include=['float64', 'int64']).columns
               if c != 'Sample_Weight']
    synthetic = []
    for g in groups:
        for y in [0, 1]:
            sub = df[(df[sensitive_col] == g) & (df[target_col] == y)]
            cnt = len(sub)
            if cnt == 0 or cnt >= max_c * 0.3:
                continue
            n_syn = min(int(max_c * 0.3) - cnt, cnt)
            if n_syn <= 0:
                continue
            samp = sub.sample(n=n_syn, replace=True, random_state=42).copy()
            for col in num_cols:
                if col in samp.columns:
                    std = sub[col].std()
                    if std > 0:
                        samp[col] = samp[col].values + np.random.normal(0, std * 0.05, n_syn)
            synthetic.append(samp)
    if synthetic:
        syn_df = pd.concat(synthetic, ignore_index=True)
        orig   = len(df)
        df     = pd.concat([df, syn_df], ignore_index=True)
        print(f'      Added {len(syn_df)} synthetic records ({orig} -> {len(df)})')
    else:
        print(f'      Groups balanced -- no augmentation needed')
    return df


def algo6_prowsyn(df, sensitive_col, target_col, k=5):
    """Algorithm 6: ProWSyn Oversampling."""
    print(f'    [Algo 6] ProWSyn Oversampling')
    num_cols  = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    feat_cols = [c for c in num_cols if c not in [target_col, 'Sample_Weight', sensitive_col]]
    if not feat_cols:
        print('      No features -- skipped')
        return df
    groups    = df[sensitive_col].dropna().unique()
    synthetic = []
    for g in groups:
        gd       = df[df[sensitive_col] == g]
        minority = gd[gd[target_col] == 1]
        majority = gd[gd[target_col] == 0]
        if len(minority) == 0 or len(majority) == 0:
            continue
        n_gen = min(max(0, len(majority) - len(minority)), len(minority))
        if n_gen == 0:
            continue
        scaler = StandardScaler()
        mf = scaler.fit_transform(minority[feat_cols].fillna(0))
        kk = min(k, len(minority) - 1)
        if kk < 1:
            continue
        nn = NearestNeighbors(n_neighbors=kk + 1)
        nn.fit(mf)
        _, indices = nn.kneighbors(mf)
        for i in range(min(n_gen, len(minority))):
            idx    = i % len(minority)
            chosen = np.random.choice(indices[idx][1:])
            alpha  = np.random.uniform(0, 1)
            new_f  = mf[idx] * alpha + mf[chosen] * (1 - alpha)
            new_row = minority.iloc[idx].copy()
            new_row[feat_cols] = scaler.inverse_transform(new_f.reshape(1, -1))[0]
            synthetic.append(new_row)
    if synthetic:
        syn_df = pd.DataFrame(synthetic)
        orig   = len(df)
        df     = pd.concat([df, syn_df], ignore_index=True)
        print(f'      Added {len(syn_df)} samples ({orig} -> {len(df)})')
    else:
        print(f'      No oversampling needed')
    return df


def algo7_counterfactual(df, sensitive_col, target_col):
    """Algorithm 7: Counterfactual Data Generation."""
    print(f'    [Algo 7] Counterfactual Data Generation')
    groups = sorted(df[sensitive_col].dropna().unique())
    if len(groups) < 2:
        print('      < 2 groups -- skipped')
        return df
    cfs = []
    for orig_val in groups:
        sub = df[df[sensitive_col] == orig_val].copy()
        for tgt_val in groups:
            if tgt_val == orig_val:
                continue
            cf = sub.copy()
            cf[sensitive_col]    = tgt_val
            cf['Is_Counterfactual'] = 1
            cfs.append(cf)
    if cfs:
        cf_df = pd.concat(cfs, ignore_index=True)
        df['Is_Counterfactual'] = 0
        orig = len(df)
        df   = pd.concat([df, cf_df], ignore_index=True)
        print(f'      Generated {len(cf_df)} counterfactuals ({orig} -> {len(df)})')
    return df


def apply_all_algorithms(df, sensitive_col, target_col):
    """Apply all 7 Stage 1 algorithms."""
    df = algo1_cchart_di_imputation(df, sensitive_col, target_col)
    df = algo2_mice(df, target_col)
    df = algo3_reweighing(df, sensitive_col, target_col)
    df = algo4_di_remover(df, sensitive_col, target_col)
    df = algo5_gan_augmentation(df, sensitive_col, target_col)
    df = algo6_prowsyn(df, sensitive_col, target_col)
    df = algo7_counterfactual(df, sensitive_col, target_col)
    return df


def final_cleanup(df):
    """Fill any remaining NaNs after augmentation."""
    for col in df.select_dtypes(include=['float64', 'int64']).columns:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mean() if not pd.isna(df[col].mean()) else 0)
    for col in df.select_dtypes(include=['object']).columns:
        if df[col].isnull().sum() > 0:
            mode = df[col].mode()
            df[col] = df[col].fillna(mode[0] if len(mode) > 0 else 'Unknown')
    return df

print('✅ All 7 bias-mitigation algorithms defined')

---
## 🟢 Stage 3: ND Folder — Preprocessing with Bias Mitigation

### 3.1 MIMIC-IV v3.1 (Hospital + ICU EHR)

- **Source**: PhysioNet MIMIC-IV v3.1
- **Chronic Disease**: ACUTE + CHRONIC — ICU treats acute events caused by chronic conditions
- **Target**: Hospital mortality (`hospital_expire_flag`)
- **Sensitive Attribute**: Gender (M/F)

**Objectives Addressed**:
- Obj 1 (RG1,RG4): Consistent mortality predictions by gender
- Obj 2 (RG1): Stage 1 pre-processing with 7 algorithms
- Obj 3 (RG2): Mortality = clinical risk target
- Obj 4 (RG3,RG4): SPD & DI audit before/after
- Obj 5 (RG3): All processing on local data

In [ ]:
def process_mimic_iv():
    print('\n' + '=' * 70)
    print('  ND DATASET 1: MIMIC-IV v3.1')
    print('  Chronic Disease: ACUTE + CHRONIC')
    print('=' * 70)

    patient_path = ND_PATHS['mimic_patients']
    adm_path     = ND_PATHS['mimic_admissions']
    icu_path     = ND_PATHS['mimic_icustays']

    if not patient_path.exists():
        print('  ⚠️  MIMIC-IV patients not found — skipping')
        return None, None

    # ── Load tables ──
    print('  Loading patients...')
    pat = pd.read_csv(patient_path)
    print(f'  Patients: {pat.shape}')

    # Demographics
    if 'gender' in pat.columns:
        pat['Gender_Encoded'] = (pat['gender'] == 'F').astype(int)  # 0=Male, 1=Female
    if 'anchor_age' in pat.columns:
        pat.rename(columns={'anchor_age': 'age'}, inplace=True)
        pat['age'] = pd.to_numeric(pat['age'], errors='coerce').clip(0, 120)

    # ── Merge admissions ──
    if adm_path.exists():
        print('  Loading admissions...')
        adm = pd.read_csv(adm_path)
        print(f'  Admissions: {adm.shape}')

        # Aggregate per patient
        agg = {'hadm_id': 'count'}
        if 'hospital_expire_flag' in adm.columns:
            agg['hospital_expire_flag'] = 'max'
        adm_feat = adm.groupby('subject_id').agg(agg).reset_index()
        adm_feat.rename(columns={'hadm_id': 'n_admissions'}, inplace=True)

        # Merge race (most common per patient)
        if 'race' in adm.columns:
            race_mode = adm.groupby('subject_id')['race'].apply(
                lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Unknown'
            ).reset_index()
            race_mode.columns = ['subject_id', 'race']
            adm_feat = adm_feat.merge(race_mode, on='subject_id', how='left')

        pat = pat.merge(adm_feat, on='subject_id', how='left')
        if 'hospital_expire_flag' in pat.columns:
            pat['Mortality'] = pat['hospital_expire_flag'].fillna(0).astype(int)
        else:
            pat['Mortality'] = 0
        print(f'  After admission merge: {pat.shape}')

    # ── Merge ICU stays ──
    if icu_path.exists():
        print('  Loading ICU stays...')
        icu = pd.read_csv(icu_path)
        icu_feat = icu.groupby('subject_id').agg(
            n_icu_stays  = ('stay_id', 'count'),
            total_icu_los= ('los', 'sum'),
            avg_icu_los  = ('los', 'mean')
        ).reset_index()
        pat = pat.merge(icu_feat, on='subject_id', how='left')
        print(f'  After ICU merge: {pat.shape}')

    # ── Build working dataframe ──
    target_col    = 'Mortality'
    sensitive_col = 'Gender_Encoded'

    # Select numeric + key features
    keep_cols = [c for c in ['subject_id', 'age', 'Gender_Encoded', 'Mortality',
                             'n_admissions', 'n_icu_stays', 'total_icu_los', 'avg_icu_los']
                 if c in pat.columns]
    df = pat[keep_cols].copy()

    # Encode race if available
    if 'race' in pat.columns:
        race_map = {'WHITE': 0, 'BLACK/AFRICAN AMERICAN': 1, 'HISPANIC/LATINO': 2,
                    'ASIAN': 3, 'OTHER': 4, 'UNKNOWN': 5}
        df['Race_Encoded'] = pat['race'].str.upper().map(
            lambda x: race_map.get(x, 5) if isinstance(x, str) else 5
        )

    # Convert to numeric
    for col in df.columns:
        if col not in ['subject_id']:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Fill NaN for counts
    for col in ['n_admissions', 'n_icu_stays', 'total_icu_los', 'avg_icu_los']:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    print(f'\n  Working dataframe: {df.shape}')
    pos = df[target_col].sum()
    print(f'  Mortality: {pos}/{len(df)} ({pos/len(df)*100:.1f}%)')
    print(f'  Gender: Male={len(df[df[sensitive_col]==0])}, Female={len(df[df[sensitive_col]==1])}')

    # ── BEFORE fairness audit ──
    print('\n  --- FAIRNESS AUDIT: BEFORE ---')
    before = fairness_audit(df, target_col, sensitive_col, 0, 1, 'Gender (Male=priv)')

    # ── Apply 7 algorithms ──
    print('\n  --- APPLYING 7 PRE-PROCESSING ALGORITHMS ---')
    df = apply_all_algorithms(df, sensitive_col, target_col)
    df = final_cleanup(df)

    # ── AFTER fairness audit ──
    df_real = df[df['Is_Counterfactual'] == 0].copy() if 'Is_Counterfactual' in df.columns else df.copy()
    print('\n  --- FAIRNESS AUDIT: AFTER ---')
    after = fairness_audit(df_real, target_col, sensitive_col, 0, 1, 'Gender (Male=priv)')

    # ── Save ──
    out_path = ND_PP / 'mimic_iv_preprocessed.csv'
    df.to_csv(out_path, index=False)
    print(f'\n  [OK] Saved: {out_path}')
    print(f'  Shape: {df.shape}, Missing: {df.isnull().sum().sum()}')
    print(f'  Before SPD: {before["SPD"]} -> After SPD: {after["SPD"]}')
    return before, after

mimic_before, mimic_after = process_mimic_iv() or (None, None)
print('\n✅ MIMIC-IV preprocessing complete')

### 3.2 eICU-CRD v2.0 (ICU Collaborative Research Database)

- **Source**: PhysioNet eICU-CRD v2.0
- **Chronic Disease**: ACUTE + CHRONIC — ICU mortality prediction
- **Target**: Hospital discharge status = Expired (mortality)
- **Sensitive Attribute**: Gender
- **Note**: Lab data (~516MB gz) skipped; using patient + diagnosis tables only

In [ ]:
def process_eicu_crd_v2():
    print('\n' + '=' * 70)
    print('  ND DATASET 2: eICU-CRD v2.0')
    print('  Chronic Disease: ACUTE + CHRONIC')
    print('=' * 70)

    patient_path = ND_PATHS['eicu2_patient']
    diag_path    = ND_PATHS['eicu2_diagnosis']

    if not patient_path.exists():
        print('  ⚠️  eICU-CRD v2.0 patient file not found — skipping')
        return None, None

    # ── Load patient table ──
    print('  Loading patient table (gzip)...')
    pat = pd.read_csv(patient_path, compression='gzip')
    print(f'  Patient records: {len(pat)} | columns: {pat.shape[1]}')

    # Age handling — eICU uses '> 89' for elderly patients
    if 'age' in pat.columns:
        pat['age'] = pat['age'].replace('> 89', 90)
        pat['age'] = pd.to_numeric(pat['age'], errors='coerce')

    # Gender encoding
    if 'gender' in pat.columns:
        pat['gender'] = pat['gender'].str.strip().str.title()
        pat['Gender_Encoded'] = (pat['gender'] == 'Female').astype(int)  # 0=Male, 1=Female
    else:
        pat['Gender_Encoded'] = 0

    # Ethnicity encoding
    if 'ethnicity' in pat.columns:
        pat['ethnicity'] = pat['ethnicity'].str.strip().str.title()
        eth_map = {'Caucasian': 0, 'African American': 1, 'Hispanic': 2,
                   'Asian': 3, 'Native American': 4}
        pat['Ethnicity_Encoded'] = pat['ethnicity'].map(eth_map).fillna(5).astype(int)

    # Mortality target
    if 'hospitaldischargestatus' in pat.columns:
        pat['Mortality'] = (pat['hospitaldischargestatus'].str.lower() == 'expired').astype(int)
    else:
        pat['Mortality'] = 0

    target_col    = 'Mortality'
    sensitive_col = 'Gender_Encoded'

    # ── Select numeric features ──
    num_feats = ['age', 'admissionheight', 'admissionweight', 'unitvisitnumber',
                 'Mortality', 'Gender_Encoded']
    existing = [c for c in num_feats if c in pat.columns]
    df = pat[existing].copy()

    if 'Ethnicity_Encoded' in pat.columns:
        df['Ethnicity_Encoded'] = pat['Ethnicity_Encoded']

    # ── Merge diagnosis features ──
    if diag_path.exists():
        print('  Loading diagnosis (gzip)...')
        diag = pd.read_csv(diag_path, compression='gzip')
        diag_feat = diag.groupby('patientunitstayid').agg(
            diagnosis_count  = ('diagnosisid', 'count'),
            unique_dx_count  = ('diagnosisstring', 'nunique')
        ).reset_index()

        if 'patientunitstayid' in pat.columns:
            df['patientunitstayid'] = pat['patientunitstayid']
            df = df.merge(diag_feat, on='patientunitstayid', how='left')
            df[['diagnosis_count', 'unique_dx_count']] = df[['diagnosis_count', 'unique_dx_count']].fillna(0)
            df.drop(columns=['patientunitstayid'], inplace=True)
        print(f'  After diagnosis merge: {df.shape}')
        del diag, diag_feat; gc.collect()

    # Convert to numeric
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    print(f'\n  Working dataframe: {df.shape}')
    pos = df[target_col].sum()
    print(f'  Mortality: {pos}/{len(df)} ({pos/len(df)*100:.1f}%)')
    print(f'  Gender: Male={len(df[df[sensitive_col]==0])}, Female={len(df[df[sensitive_col]==1])}')

    # ── BEFORE fairness audit ──
    print('\n  --- FAIRNESS AUDIT: BEFORE ---')
    before = fairness_audit(df, target_col, sensitive_col, 0, 1, 'Gender (Male=priv)')

    # ── Apply 7 algorithms ──
    print('\n  --- APPLYING 7 PRE-PROCESSING ALGORITHMS ---')
    df = apply_all_algorithms(df, sensitive_col, target_col)
    df = final_cleanup(df)

    # ── AFTER fairness audit ──
    df_real = df[df['Is_Counterfactual'] == 0].copy() if 'Is_Counterfactual' in df.columns else df.copy()
    print('\n  --- FAIRNESS AUDIT: AFTER ---')
    after = fairness_audit(df_real, target_col, sensitive_col, 0, 1, 'Gender (Male=priv)')

    # ── Save ──
    out_path = ND_PP / 'eicu_crd_v2_preprocessed.csv'
    df.to_csv(out_path, index=False)
    print(f'\n  [OK] Saved: {out_path}')
    print(f'  Shape: {df.shape}, Missing: {df.isnull().sum().sum()}')
    print(f'  Before SPD: {before["SPD"]} -> After SPD: {after["SPD"]}')
    return before, after

eicu2_before, eicu2_after = process_eicu_crd_v2() or (None, None)
print('\n✅ eICU-CRD v2.0 preprocessing complete')

### 3.3 PTB-XL ECG Dataset

- **Source**: PhysioNet PTB-XL v1.0.3 + Comprehensive Feature Dataset v1.0.1
- **Chronic Disease**: YES — Cardiac arrhythmias / heart conditions are chronic
- **Target**: Abnormal ECG (1) vs Normal (0) from SCP diagnostic codes
- **Sensitive Attribute**: Sex (0=Female, 1=Male)

In [ ]:
def process_ptbxl():
    print('\n' + '=' * 70)
    print('  ND DATASET 3: PTB-XL ECG Dataset')
    print('  Chronic Disease: YES — Cardiac conditions')
    print('=' * 70)

    db_path   = ND_PATHS['ptbxl_db']
    scp_path  = ND_PATHS['ptbxl_scp']
    feat_path = ND_PATHS['ptbxl_features']

    if not db_path.exists():
        print('  ⚠️  PTB-XL database CSV not found — skipping')
        return None, None

    # ── Load database ──
    print('  Loading PTB-XL database...')
    df = pd.read_csv(db_path)
    print(f'  Records: {df.shape}')

    # Parse SCP code dictionaries
    def parse_scp(scp_str):
        try:
            return ast.literal_eval(scp_str)
        except:
            return {}

    df['scp_dict'] = df['scp_codes'].apply(parse_scp)

    # Binary label: 1=abnormal, 0=normal
    def is_abnormal(scp_dict):
        if not scp_dict:
            return np.nan
        return 0 if set(scp_dict.keys()) <= {'NORM', 'SR'} else 1

    df['ECG_Abnormal'] = df['scp_dict'].apply(is_abnormal)

    # Binary flags for top diagnoses
    top_dx = ['NORM', 'MI', 'STTC', 'CD', 'HYP', 'AFIB', 'IVCD', 'LVH', 'CLBBB', 'CRBBB']
    for dx in top_dx:
        df[f'dx_{dx}'] = df['scp_dict'].apply(lambda d: 1 if dx in d and d[dx] > 0 else 0)

    # Demographics
    if 'age' in df.columns:
        df['age'] = pd.to_numeric(df['age'], errors='coerce')
        df.loc[df['age'] > 120, 'age'] = np.nan  # de-identified ages
        df['age'].fillna(df['age'].median(), inplace=True)

    # Sex encoding (0=Female, 1=Male — use as sensitive attr)
    if 'sex' in df.columns:
        df['Sex_Encoded'] = df['sex'].astype(int)

    # Quality flags
    for col in ['baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems']:
        if col in df.columns:
            df[col] = df[col].notna().astype(int)

    # ── Optionally merge 12SL feature dataset ──
    if feat_path.exists():
        print('  Merging PTB-XL comprehensive features...')
        try:
            feats = pd.read_csv(feat_path)
            join_col = feats.columns[0]
            feat_num = feats.select_dtypes(include=np.number).copy()
            feat_num[join_col] = feats[join_col]
            if join_col in df.columns:
                df = df.merge(feat_num, on=join_col, how='left', suffixes=('', '_feat'))
                print(f'  ECG features merged → {df.shape}')
        except Exception as e:
            print(f'  ⚠️  Feature merge failed: {e}')

    # ── Drop raw string/path columns ──
    df.drop(columns=['scp_dict', 'scp_codes', 'report', 'filename_lr', 'filename_hr'],
            errors='ignore', inplace=True)

    # ── Drop high-missing columns ──
    miss_pct = df.isnull().mean()
    drop_cols = miss_pct[miss_pct > 0.7].index.tolist()
    if drop_cols:
        print(f'  Dropping {len(drop_cols)} high-missing columns')
        df.drop(columns=drop_cols, inplace=True)

    # ── Select working columns ──
    target_col    = 'ECG_Abnormal'
    sensitive_col = 'Sex_Encoded'

    # Keep only numeric + target + sensitive
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    df = df[num_cols].copy()

    # Impute remaining numeric
    imp_cols = [c for c in df.columns if c not in [target_col, sensitive_col]]
    if df[imp_cols].isnull().any().any():
        df[imp_cols] = SimpleImputer(strategy='median').fit_transform(df[imp_cols])

    print(f'\n  Working dataframe: {df.shape}')
    pos = df[target_col].sum()
    print(f'  ECG_Abnormal: {pos}/{len(df)} ({pos/len(df)*100:.1f}%)')
    print(f'  Sex: Female={len(df[df[sensitive_col]==0])}, Male={len(df[df[sensitive_col]==1])}')

    # ── BEFORE fairness audit ──
    print('\n  --- FAIRNESS AUDIT: BEFORE ---')
    before = fairness_audit(df, target_col, sensitive_col, 1, 0, 'Sex (Male=priv)')

    # ── Apply 7 algorithms ──
    print('\n  --- APPLYING 7 PRE-PROCESSING ALGORITHMS ---')
    df = apply_all_algorithms(df, sensitive_col, target_col)
    df = final_cleanup(df)

    # ── AFTER fairness audit ──
    df_real = df[df['Is_Counterfactual'] == 0].copy() if 'Is_Counterfactual' in df.columns else df.copy()
    print('\n  --- FAIRNESS AUDIT: AFTER ---')
    after = fairness_audit(df_real, target_col, sensitive_col, 1, 0, 'Sex (Male=priv)')

    # ── Save ──
    out_path = ND_PP / 'ptbxl_preprocessed.csv'
    df.to_csv(out_path, index=False)
    print(f'\n  [OK] Saved: {out_path}')
    print(f'  Shape: {df.shape}, Missing: {df.isnull().sum().sum()}')
    print(f'  Before SPD: {before["SPD"]} -> After SPD: {after["SPD"]}')
    return before, after

ptbxl_before, ptbxl_after = process_ptbxl() or (None, None)
print('\n✅ PTB-XL ECG preprocessing complete')

### 3.4 HAM10000 Skin Lesion Dataset

- **Source**: Harvard Dataverse / ISIC 2018
- **Chronic Disease**: YES — Skin cancer / melanoma can be chronic
- **Target**: Melanoma (dx == 'mel') binary classification
- **Sensitive Attribute**: Age group (median split)

In [ ]:
def process_ham10000():
    print('\n' + '=' * 70)
    print('  ND DATASET 4: HAM10000 Skin Lesion')
    print('  Chronic Disease: YES — Cancer (melanoma)')
    print('=' * 70)

    meta_path = ND_PATHS['ham_metadata']
    gt_path   = ND_PATHS['ham_gt']

    if not meta_path.exists():
        print('  ⚠️  HAM10000 metadata not found — skipping')
        return None, None

    # HAM10000_metadata may have no file extension
    print('  Loading metadata...')
    try:
        df = pd.read_csv(meta_path)
    except Exception:
        try:
            df = pd.read_csv(str(meta_path) + '.csv')
        except Exception as e:
            print(f'  ⚠️  Could not read HAM10000 metadata: {e}')
            return None, None

    print(f'  Metadata shape: {df.shape}')
    print(f'  Columns: {df.columns.tolist()}')

    # ── Lesion class encoding ──
    lesion_labels = {'mel': 0, 'nv': 1, 'bcc': 2, 'akiec': 3, 'bkl': 4, 'df': 5, 'vasc': 6}
    if 'dx' in df.columns:
        df['dx_code']      = df['dx'].map(lesion_labels)
        df['Melanoma']     = (df['dx'] == 'mel').astype(int)
        df['is_malignant'] = df['dx'].isin(['mel', 'bcc', 'akiec']).astype(int)
        print(f'  Diagnosis counts:\n{df["dx"].value_counts()}')

    # Encode confirmation method
    if 'dx_type' in df.columns:
        df['dx_type_enc'] = LabelEncoder().fit_transform(df['dx_type'].fillna('unknown'))

    # Encode localization
    if 'localization' in df.columns:
        df['localization_enc'] = LabelEncoder().fit_transform(df['localization'].fillna('unknown'))

    # Age
    if 'age' in df.columns:
        df['age'] = pd.to_numeric(df['age'], errors='coerce')
        df['age'].fillna(df['age'].median(), inplace=True)

    # Sensitive attribute: Age group (median split)
    age_median = df['age'].median()
    df['Age_Group'] = (df['age'] >= age_median).astype(int)  # 1=older, 0=younger

    # Sex encoding
    if 'sex' in df.columns:
        df['Sex_Encoded'] = df['sex'].map({'male': 1, 'female': 0, 'unknown': -1})
        df['Sex_Encoded'] = df['Sex_Encoded'].fillna(-1).astype(int)

    target_col    = 'Melanoma'
    sensitive_col = 'Age_Group'

    # ── Keep numeric columns ──
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    df_work = df[num_cols].copy()

    # Impute remaining
    imp_cols = [c for c in df_work.columns if c not in [target_col, sensitive_col]]
    if df_work[imp_cols].isnull().any().any():
        df_work[imp_cols] = SimpleImputer(strategy='median').fit_transform(df_work[imp_cols])

    print(f'\n  Working dataframe: {df_work.shape}')
    pos = df_work[target_col].sum()
    print(f'  Melanoma: {pos}/{len(df_work)} ({pos/len(df_work)*100:.1f}%)')
    print(f'  Age split: Younger(<{age_median:.0f})={len(df_work[df_work[sensitive_col]==0])}, '
          f'Older(>={age_median:.0f})={len(df_work[df_work[sensitive_col]==1])}')

    # ── BEFORE fairness audit ──
    print('\n  --- FAIRNESS AUDIT: BEFORE ---')
    before = fairness_audit(df_work, target_col, sensitive_col, 0, 1, 'Age_Group (Young=priv)')

    # ── Apply 7 algorithms ──
    print('\n  --- APPLYING 7 PRE-PROCESSING ALGORITHMS ---')
    df_work = apply_all_algorithms(df_work, sensitive_col, target_col)
    df_work = final_cleanup(df_work)

    # ── AFTER fairness audit ──
    df_real = df_work[df_work['Is_Counterfactual'] == 0].copy() if 'Is_Counterfactual' in df_work.columns else df_work.copy()
    print('\n  --- FAIRNESS AUDIT: AFTER ---')
    after = fairness_audit(df_real, target_col, sensitive_col, 0, 1, 'Age_Group (Young=priv)')

    # ── Save ──
    out_path = ND_PP / 'ham10000_preprocessed.csv'
    df_work.to_csv(out_path, index=False)
    print(f'\n  [OK] Saved: {out_path}')
    print(f'  Shape: {df_work.shape}, Missing: {df_work.isnull().sum().sum()}')
    print(f'  Before SPD: {before["SPD"]} -> After SPD: {after["SPD"]}')
    return before, after

ham_before, ham_after = process_ham10000() or (None, None)
print('\n✅ HAM10000 preprocessing complete')

### 3.5 Heart Failure EHR Dataset

- **Source**: PhysioNet — Hospitalized patients with heart failure
- **Chronic Disease**: YES — Heart failure is a chronic, progressive condition
- **Target**: Death/outcome (auto-detected from columns)
- **Sensitive Attribute**: Age group (median split) or sex

In [ ]:
def process_heart_failure_ehr():
    print('\n' + '=' * 70)
    print('  ND DATASET 5: Heart Failure EHR')
    print('  Chronic Disease: YES — Heart failure')
    print('=' * 70)

    dat_path    = ND_PATHS['hf_dat']
    dat_md_path = ND_PATHS['hf_dat_md']

    if not dat_path.exists():
        print('  ⚠️  Heart Failure EHR dat.csv not found — skipping')
        return None, None

    print('  Loading dat.csv...')
    df = pd.read_csv(dat_path)
    print(f'  Raw shape: {df.shape}')
    print(f'  Columns (first 20): {df.columns.tolist()[:20]}')

    # Load data dictionary if available
    if dat_md_path.exists():
        md = pd.read_csv(dat_md_path)
        print(f'  Data dictionary: {md.shape}')

    # ── Find target column ──
    target_candidates = ['death', 'deceased', 'outcome', 'DEATH', 'event', 'expire',
                         'death_inhosp', 'death_28day', 'death_90day', 'death_1year']
    target_col = None
    for c in target_candidates:
        if c in df.columns:
            target_col = c
            break
    # Case-insensitive search
    if target_col is None:
        for c in df.columns:
            if any(kw in c.lower() for kw in ['death', 'expire', 'outcome', 'deceased']):
                target_col = c
                break

    if target_col:
        df['HF_Mortality'] = pd.to_numeric(df[target_col], errors='coerce').fillna(0).astype(int)
        print(f'  Target column found: "{target_col}" → {df["HF_Mortality"].value_counts().to_dict()}')
    else:
        # Try to find any binary column
        print('  ⚠️  No clear target column found — checking binary columns...')
        for col in df.columns:
            try:
                vals = pd.to_numeric(df[col], errors='coerce').dropna()
                if vals.isin([0, 1]).all() and vals.nunique() == 2:
                    df['HF_Mortality'] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
                    print(f'  Using "{col}" as target: {df["HF_Mortality"].value_counts().to_dict()}')
                    break
            except:
                continue
        else:
            df['HF_Mortality'] = 0
            print('  ⚠️  No suitable target found — using 0')

    target_col = 'HF_Mortality'

    # ── Convert all to numeric ──
    for col in df.columns:
        if col != target_col:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # ── Drop high-missing columns (>50%) ──
    miss_pct = df.isnull().mean()
    drop_cols = miss_pct[miss_pct > 0.5].index.tolist()
    if drop_cols:
        print(f'  Dropping {len(drop_cols)} high-missing columns')
        df.drop(columns=drop_cols, inplace=True)

    # ── Sensitive attribute: Age group ──
    age_col_candidates = [c for c in df.columns if 'age' in c.lower()]
    if age_col_candidates:
        age_col = age_col_candidates[0]
        if age_col != 'age':
            df.rename(columns={age_col: 'age'}, inplace=True)
        age_median = df['age'].median()
        df['Age_Group'] = (df['age'] >= age_median).astype(int)  # 1=older, 0=younger
        sensitive_col = 'Age_Group'
    elif 'sex' in df.columns or 'gender' in df.columns:
        sex_col = 'sex' if 'sex' in df.columns else 'gender'
        df['Sex_Encoded'] = df[sex_col].astype(int)
        sensitive_col = 'Sex_Encoded'
        age_median = 0
    else:
        # Fallback: use first numeric column median split
        num_cols = df.select_dtypes(include=np.number).columns.difference([target_col])
        if len(num_cols) > 0:
            df['Group'] = (df[num_cols[0]] >= df[num_cols[0]].median()).astype(int)
            sensitive_col = 'Group'
        else:
            print('  ⚠️  No suitable sensitive attribute found')
            return None, None
        age_median = 0

    # ── Impute numeric features ──
    num_cols = df.select_dtypes(include=np.number).columns.difference([target_col, sensitive_col])
    if df[num_cols].isnull().any().any():
        df[num_cols] = SimpleImputer(strategy='median').fit_transform(df[num_cols])

    print(f'\n  Working dataframe: {df.shape}')
    pos = df[target_col].sum()
    print(f'  HF_Mortality: {pos}/{len(df)} ({pos/len(df)*100:.1f}%)')

    # ── BEFORE fairness audit ──
    print('\n  --- FAIRNESS AUDIT: BEFORE ---')
    before = fairness_audit(df, target_col, sensitive_col, 0, 1, f'{sensitive_col} (group=0 priv)')

    # ── Apply 7 algorithms ──
    print('\n  --- APPLYING 7 PRE-PROCESSING ALGORITHMS ---')
    df = apply_all_algorithms(df, sensitive_col, target_col)
    df = final_cleanup(df)

    # ── AFTER fairness audit ──
    df_real = df[df['Is_Counterfactual'] == 0].copy() if 'Is_Counterfactual' in df.columns else df.copy()
    print('\n  --- FAIRNESS AUDIT: AFTER ---')
    after = fairness_audit(df_real, target_col, sensitive_col, 0, 1, f'{sensitive_col} (group=0 priv)')

    # ── Save ──
    out_path = ND_PP / 'heart_failure_ehr_preprocessed.csv'
    df.to_csv(out_path, index=False)
    print(f'\n  [OK] Saved: {out_path}')
    print(f'  Shape: {df.shape}, Missing: {df.isnull().sum().sum()}')
    print(f'  Before SPD: {before["SPD"]} -> After SPD: {after["SPD"]}')
    return before, after

hf_before, hf_after = process_heart_failure_ehr() or (None, None)
print('\n✅ Heart Failure EHR preprocessing complete')

### 3.6 MIMIC-CXR Reports (Metadata Only)

- **Source**: PhysioNet MIMIC-CXR
- **Note**: This is a **text-based** dataset (radiology reports). The 7 bias-mitigation algorithms are designed for tabular data, so we extract metadata statistics only.
- **No bias algorithms applied** — text data requires NLP-specific debiasing

In [ ]:
def process_mimic_cxr_metadata():
    print('\n' + '=' * 70)
    print('  ND DATASET 6: MIMIC-CXR Reports (Metadata Only)')
    print('  Note: Text data — no bias algorithms applied')
    print('=' * 70)

    cxr_dir = ND_PATHS['mimic_cxr_dir']

    if not cxr_dir.exists():
        print('  ⚠️  MIMIC-CXR reports directory not found — skipping')
        return None

    # ── Count reports per patient folder ──
    print('  Scanning report directory structure (optimized)...')
    
    report_counts = []
    total_reports = 0
    total_subjects = 0
    
    import os
    cxr_dir_abs = os.path.abspath(cxr_dir)
    
    for root, dirs, files in os.walk(cxr_dir):
        parent = os.path.dirname(root)
        grandparent = os.path.dirname(parent)
        if os.path.abspath(grandparent) == cxr_dir_abs:
            patient_group = os.path.basename(parent)
            subject_id = os.path.basename(root)
            n_reports = len([f for f in files if f.endswith('.txt')])
            total_reports += n_reports
            total_subjects += 1
            report_counts.append({
                'patient_group': patient_group,
                'subject_id':    subject_id,
                'n_reports':     n_reports
            })

    print(f'  Total subjects: {total_subjects}')
    print(f'  Total reports:  {total_reports}')

    if report_counts:
        meta_df = pd.DataFrame(report_counts)
        print(f'\n  Reports per subject:')
        print(f'    Mean:   {meta_df["n_reports"].mean():.1f}')
        print(f'    Median: {meta_df["n_reports"].median():.0f}')
        print(f'    Max:    {meta_df["n_reports"].max()}')
        print(f'    Min:    {meta_df["n_reports"].min()}')

        # Save metadata
        out_path = ND_PP / 'mimic_cxr_metadata.csv'
        meta_df.to_csv(out_path, index=False)
        print(f'\n  [OK] Saved metadata: {out_path}')
        return meta_df
    return None

cxr_meta = process_mimic_cxr_metadata()
print('\n✅ MIMIC-CXR metadata extraction complete')


### 3.7 ND Preprocessing Summary

In [ ]:
# ── ND Preprocessing Summary ──
print('=' * 70)
print('  ND FOLDER — PREPROCESSING SUMMARY')
print('=' * 70)

nd_results = {
    'MIMIC-IV v3.1':       {'before': mimic_before, 'after': mimic_after,  'chronic': 'ACUTE+CHRONIC'},
    'eICU-CRD v2.0':       {'before': eicu2_before, 'after': eicu2_after,  'chronic': 'ACUTE+CHRONIC'},
    'PTB-XL ECG':          {'before': ptbxl_before, 'after': ptbxl_after,  'chronic': 'YES'},
    'HAM10000':            {'before': ham_before,   'after': ham_after,    'chronic': 'YES'},
    'Heart Failure EHR':   {'before': hf_before,    'after': hf_after,    'chronic': 'YES'},
}

print(f'\n  {"Dataset":<20} {"Chronic?":<15} {"SPD Before":>12} {"SPD After":>12} {"DI Before":>12} {"DI After":>12}')
print('  ' + '-' * 85)
for name, r in nd_results.items():
    if r['before'] is not None and r['after'] is not None:
        print(f'  {name:<20} {r["chronic"]:<15} '
              f'{str(r["before"]["SPD"]):>12} {str(r["after"]["SPD"]):>12} '
              f'{str(r["before"]["DI"]):>12} {str(r["after"]["DI"]):>12}')
    else:
        print(f'  {name:<20} {r["chronic"]:<15} {"SKIPPED":>12} {"":>12} {"":>12} {"":>12}')

print('\n  Note: MIMIC-CXR (text reports) processed as metadata only — no bias algorithms applied')

---
## 📂 Stage 4: Load Data_Set Preprocessed Outputs

Load the 6 already-preprocessed datasets from `Data_Set/preprocessed_output/`

In [ ]:
# ── Load Data_Set preprocessed outputs ──
print('📂 Loading Data_Set preprocessed datasets...')
print('=' * 65)

ds_files = {
    'Breast_Cancer'  : DS_PP / 'breast_cancer_preprocessed.csv',
    'CDC_Diabetes'   : DS_PP / 'cdc_diabetes_preprocessed.csv',
    'CKD'            : DS_PP / 'ckd_preprocessed.csv',
    'eICU_original'  : DS_PP / 'eicu_preprocessed.csv',
    'Heart_Disease'  : DS_PP / 'heart_disease_preprocessed.csv',
    'NHANES'         : DS_PP / 'NHANES_preprocessed.csv',
}

ds_loaded = {}
for name, path in ds_files.items():
    if path.exists():
        df = pd.read_csv(path)
        ds_loaded[name] = df
        print(f'  ✅ {name:<20} {str(df.shape):<18} {df.isnull().mean().mean()*100:.1f}% missing')
    else:
        print(f'  ❌ {name:<20} NOT FOUND at {path}')

# ── Load ND preprocessed outputs ──
print('\n📂 Loading ND preprocessed datasets...')
print('=' * 65)

nd_files = {
    'MIMIC_IV'           : ND_PP / 'mimic_iv_preprocessed.csv',
    'eICU_CRD_v2'        : ND_PP / 'eicu_crd_v2_preprocessed.csv',
    'PTB_XL_ECG'         : ND_PP / 'ptbxl_preprocessed.csv',
    'HAM10000'           : ND_PP / 'ham10000_preprocessed.csv',
    'HeartFailure_EHR'   : ND_PP / 'heart_failure_ehr_preprocessed.csv',
}

nd_loaded = {}
for name, path in nd_files.items():
    if path.exists():
        df = pd.read_csv(path)
        nd_loaded[name] = df
        print(f'  ✅ {name:<20} {str(df.shape):<18} {df.isnull().mean().mean()*100:.1f}% missing')
    else:
        print(f'  ❌ {name:<20} NOT FOUND at {path}')

print(f'\n✅ Loaded {len(ds_loaded)} Data_Set + {len(nd_loaded)} ND = {len(ds_loaded)+len(nd_loaded)} total datasets')

---
## 🔗 Stage 5: Cross-Dataset Integration

### 5.1 Harmonize to Unified Schema

In [ ]:
# ── Common schema ──
COMMON_SCHEMA = ['dataset_source', 'domain', 'target', 'age', 'age_group', 'gender']

# ── Domain mapping ──
DOMAIN_MAP = {
    # Data_Set
    'Breast_Cancer'  : 'oncology',
    'CDC_Diabetes'   : 'metabolic',
    'CKD'            : 'renal',
    'eICU_original'  : 'icu',
    'Heart_Disease'  : 'cardiovascular',
    'NHANES'         : 'population_health',
    # ND
    'MIMIC_IV'         : 'icu',
    'eICU_CRD_v2'      : 'icu',
    'PTB_XL_ECG'       : 'ecg',
    'HAM10000'         : 'dermatology',
    'HeartFailure_EHR' : 'cardiovascular',
}

# ── Target column mapping ──
TARGET_MAP = {
    'Breast_Cancer'    : 'Cancer_Risk',
    'CDC_Diabetes'     : 'Diabetes_binary',
    'CKD'              : 'CKD_Risk',
    'eICU_original'    : 'Mortality',
    'Heart_Disease'    : 'Heart_Disease',
    'NHANES'           : None,  # survey data, no clear target
    'MIMIC_IV'         : 'Mortality',
    'eICU_CRD_v2'      : 'Mortality',
    'PTB_XL_ECG'       : 'ECG_Abnormal',
    'HAM10000'         : 'Melanoma',
    'HeartFailure_EHR' : 'HF_Mortality',
}


def harmonize_dataset(df, name):
    """Harmonize a dataset to the common unified schema."""
    out = pd.DataFrame(index=range(len(df)))
    out['dataset_source'] = name
    out['domain']         = DOMAIN_MAP.get(name, 'unknown')

    # Target
    target_col = TARGET_MAP.get(name)
    if target_col and target_col in df.columns:
        out['target'] = pd.to_numeric(df[target_col].values, errors='coerce')
    elif 'target' in df.columns:
        out['target'] = pd.to_numeric(df['target'].values, errors='coerce')
    else:
        out['target'] = np.nan

    # Age
    age_found = False
    for age_c in ['age', 'Age', 'age_clean', 'anchor_age']:
        if age_c in df.columns:
            out['age'] = pd.to_numeric(df[age_c].values, errors='coerce')
            age_found = True
            break
    if not age_found:
        out['age'] = np.nan

    # Age group
    if 'age_group' in df.columns:
        out['age_group'] = df['age_group'].astype(str).values
    elif 'Age_Group' in df.columns:
        out['age_group'] = df['Age_Group'].astype(str).values
    else:
        out['age_group'] = pd.cut(out['age'], bins=[0, 40, 60, 75, 120],
                                  labels=['young', 'middle', 'senior', 'elderly'],
                                  include_lowest=True).astype(str)

    # Gender
    gender_found = False
    for g_col in ['gender', 'Gender_Encoded', 'Sex_Encoded', 'sex', 'Sex']:
        if g_col in df.columns:
            g = df[g_col].astype(str).str.strip().str.lower()
            out['gender'] = g.map(
                lambda x: 'Male'    if x in ['1', '1.0', 'm', 'male', 'man', '0']  else
                          'Female'  if x in ['2', '0.0', 'f', 'female', 'woman', '1'] else 'Unknown'
            )
            # Fix for Gender_Encoded where 0=Male, 1=Female
            if g_col == 'Gender_Encoded':
                out['gender'] = df[g_col].map({0: 'Male', 1: 'Female'}).fillna('Unknown')
            elif g_col == 'Sex_Encoded':
                out['gender'] = df[g_col].map({0: 'Female', 1: 'Male'}).fillna('Unknown')
            gender_found = True
            break
    if not gender_found:
        out['gender'] = 'Unknown'

    # Carry all original columns
    for col in df.columns:
        if col not in COMMON_SCHEMA and col not in out.columns:
            out[col] = df[col].values

    target_valid = out['target'].notna().sum()
    print(f'  [{name:<22}] → {out.shape} | target_valid: {target_valid:,} | missing: {out.isnull().mean().mean()*100:.1f}%')
    return out


# ── Harmonize ALL datasets ──
print('📐 Harmonizing all datasets to unified schema...')
print('=' * 75)

all_datasets = {**ds_loaded, **nd_loaded}
harmonized   = {}

for name, df in all_datasets.items():
    harmonized[name] = harmonize_dataset(df, name)

print(f'\n✅ {len(harmonized)} datasets harmonized')

### 5.2 Domain-Level Integration

In [ ]:
# ── Group by clinical domain ──
domain_groups = {}
for name, df in harmonized.items():
    domain = df['domain'].iloc[0] if 'domain' in df.columns and len(df) > 0 else 'unknown'
    if domain not in domain_groups:
        domain_groups[domain] = []
    domain_groups[domain].append((name, df))

print('📂 Domain Groups:')
print('=' * 60)
for domain, datasets in domain_groups.items():
    total = sum(len(df) for _, df in datasets)
    names = [f'{n}({len(df):,})' for n, df in datasets]
    print(f'  {domain:<25}: {total:>10,} records')
    for n in names:
        print(f'    ↳ {n}')

# ── Integrate within each domain ──
print('\n🔗 Integrating by domain...')
print('=' * 60)
domain_dfs = {}
for domain, datasets in domain_groups.items():
    frames = [df for _, df in datasets]
    integrated = pd.concat(frames, axis=0, ignore_index=True, sort=False)
    front_cols = [c for c in COMMON_SCHEMA if c in integrated.columns]
    other_cols = [c for c in integrated.columns if c not in front_cols]
    domain_dfs[domain] = integrated[front_cols + other_cols]
    miss = integrated.isnull().mean().mean() * 100
    print(f'  [{domain:<20}] → {integrated.shape} | missing: {miss:.1f}%')

print(f'\n✅ {len(domain_dfs)} domain-integrated DataFrames created')

### 5.3 Full Cross-Dataset Integration

In [ ]:
# ── Full integration of ALL datasets ──
all_frames = list(harmonized.values())
full_integrated = pd.concat(all_frames, axis=0, ignore_index=True, sort=False)

# Reorder: schema columns first
front_cols = [c for c in COMMON_SCHEMA if c in full_integrated.columns]
other_cols = [c for c in full_integrated.columns if c not in front_cols]
full_integrated = full_integrated[front_cols + other_cols]

print('=' * 70)
print('🏥 FULL INTEGRATED DATASET — SUMMARY')
print('=' * 70)
print(f'   Total records  : {len(full_integrated):>12,}')
print(f'   Total features : {full_integrated.shape[1]:>12,}')
print(f'   Overall missing: {full_integrated.isnull().mean().mean()*100:>11.1f}%')
print()
print('   Records per dataset source:')
print(full_integrated['dataset_source'].value_counts().to_string())
print()
print('   Records per clinical domain:')
print(full_integrated['domain'].value_counts().to_string())

---
## 📊 Stage 6: Quality Validation & Visualization

In [ ]:
# ── Dataset Statistics Summary Table ──
summary_rows = []
for name, df in harmonized.items():
    row = {
        'Dataset'    : name,
        'Source'     : 'Data_Set' if name in ds_loaded else 'ND',
        'Domain'     : df['domain'].iloc[0] if 'domain' in df.columns else 'N/A',
        'Records'    : len(df),
        'Features'   : df.shape[1],
        'Missing%'   : f'{df.isnull().mean().mean()*100:.1f}%',
        'Target_Rate': f'{df["target"].mean()*100:.1f}%' if 'target' in df.columns and df['target'].notna().any() else 'N/A',
        'Has_Age'    : '✅' if df['age'].notna().sum() > 0   else '❌',
        'Has_Gender' : '✅' if 'gender' in df.columns         else '❌',
        'Has_Target' : '✅' if df['target'].notna().sum() > 0 else '❌',
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print('📋 Complete Dataset Integration Summary')
print('=' * 120)
print(summary_df.to_string(index=False))

In [ ]:
# ── Comprehensive Dashboard Visualization ──
fig = plt.figure(figsize=(22, 18))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.38)
fig.suptitle('🏥 Multi-Dataset Integration — Quality & Overview Dashboard', fontsize=16, fontweight='bold')

# 1. Records per dataset
ax1 = fig.add_subplot(gs[0, 0])
rec = full_integrated['dataset_source'].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(rec)))
bars = ax1.barh(range(len(rec)), rec.values, color=colors, edgecolor='white')
ax1.set_yticks(range(len(rec)))
ax1.set_yticklabels([t[:22] for t in rec.index], fontsize=8)
ax1.set_title('Records per Dataset', fontweight='bold')
ax1.set_xlabel('Count')
for bar, val in zip(bars, rec.values):
    ax1.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2,
             f'{val:,}', va='center', fontsize=7)

# 2. Missing % per dataset
ax2 = fig.add_subplot(gs[0, 1])
miss_by_ds = {n: df.isnull().mean().mean()*100 for n, df in harmonized.items()}
max_miss = max(miss_by_ds.values()) if miss_by_ds.values() else 1
colors2 = [plt.cm.RdYlGn_r(v / (max_miss + 1e-6)) for v in miss_by_ds.values()]
ax2.barh(list(miss_by_ds.keys()), list(miss_by_ds.values()), color=colors2, edgecolor='white')
ax2.set_title('Missing Data % per Dataset', fontweight='bold')
ax2.set_xlabel('Missing %')
ax2.axvline(30, ls='--', color='red', alpha=0.7, label='30% threshold')
ax2.set_yticklabels([k[:18] for k in miss_by_ds.keys()], fontsize=8)
ax2.legend(fontsize=8)

# 3. Target (positive class) rate
ax3 = fig.add_subplot(gs[0, 2])
target_rates = {n: df['target'].mean()*100 for n, df in harmonized.items()
                if 'target' in df.columns and df['target'].notna().any()}
if target_rates:
    ax3.bar(range(len(target_rates)), list(target_rates.values()),
            color=plt.cm.Spectral(np.linspace(0.1, 0.9, len(target_rates))), edgecolor='white')
    ax3.set_xticks(range(len(target_rates)))
    ax3.set_xticklabels([k[:14] for k in target_rates.keys()], rotation=45, ha='right', fontsize=7)
    ax3.set_title('Positive Class Rate (%)', fontweight='bold')
    ax3.set_ylabel('%')
    ax3.axhline(50, ls='--', color='gray', alpha=0.5, label='50%')
    ax3.legend(fontsize=8)

# 4. Age distribution across all datasets
ax4 = fig.add_subplot(gs[1, 0])
age_data = full_integrated['age'].dropna()
if len(age_data) > 0:
    ax4.hist(age_data, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax4.axvline(age_data.median(), color='red', ls='--', label=f'Median: {age_data.median():.0f}')
    ax4.set_title('Age Distribution (All Datasets)', fontweight='bold')
    ax4.set_xlabel('Age'); ax4.set_ylabel('Count')
    ax4.legend(fontsize=9)

# 5. Gender distribution
ax5 = fig.add_subplot(gs[1, 1])
gender_counts = full_integrated['gender'].value_counts().head(5)
gender_colors = ['#4ECDC4', '#FF6B6B', '#95A5A6', '#F39C12', '#8E44AD']
ax5.pie(gender_counts.values, labels=gender_counts.index,
        autopct='%1.1f%%', colors=gender_colors[:len(gender_counts)])
ax5.set_title('Gender Distribution (All Datasets)', fontweight='bold')

# 6. Domain distribution
ax6 = fig.add_subplot(gs[1, 2])
domain_counts = full_integrated['domain'].value_counts()
ax6.bar(range(len(domain_counts)), domain_counts.values,
        color=plt.cm.tab10(np.linspace(0, 1, len(domain_counts))), edgecolor='white')
ax6.set_xticks(range(len(domain_counts)))
ax6.set_xticklabels(domain_counts.index, rotation=30, ha='right', fontsize=9)
ax6.set_title('Records by Clinical Domain', fontweight='bold')
ax6.set_ylabel('Records')

# 7. Data source (Data_Set vs ND)
ax7 = fig.add_subplot(gs[2, 0])
source_type = full_integrated['dataset_source'].map(
    lambda x: 'Data_Set (Original)' if x in ds_loaded else 'ND (New)'
).value_counts()
ax7.pie(source_type.values, labels=source_type.index, autopct='%1.1f%%',
        colors=['#3498DB', '#E74C3C'], startangle=90)
ax7.set_title('Records: Data_Set vs ND', fontweight='bold')

# 8. Bias audit: Target rate by gender
ax8 = fig.add_subplot(gs[2, 1:])
bias_rows = []
for name, df in harmonized.items():
    if 'target' in df.columns and 'gender' in df.columns:
        valid_df = df[df['target'].notna() & (df['gender'] != 'Unknown')]
        if len(valid_df) > 0:
            grp = valid_df.groupby('gender')['target'].agg(['mean', 'count'])
            for gender, row in grp.iterrows():
                if row['count'] > 10:
                    bias_rows.append({'Dataset': name[:16], 'Gender': gender,
                                      'Rate': row['mean'], 'Count': int(row['count'])})

if bias_rows:
    bias_df = pd.DataFrame(bias_rows)
    pivot   = bias_df.pivot_table(values='Rate', index='Dataset', columns='Gender', aggfunc='mean')
    pivot.plot(kind='bar', ax=ax8, colormap='Set2', edgecolor='white')
    ax8.set_title('Target Rate by Gender per Dataset', fontweight='bold', fontsize=11)
    ax8.set_ylabel('Target Rate')
    ax8.tick_params(axis='x', rotation=35)
    ax8.legend(title='Gender', fontsize=8)

plt.savefig(OUT_DIR / 'plots' / 'full_integration_dashboard.png', bbox_inches='tight', dpi=120)
plt.show()
print('✅ Integration dashboard saved')

In [ ]:
# ── ND Fairness Before/After Comparison ──
print('⚖️  ND Datasets — Fairness Before/After Comparison')
print('=' * 80)

fairness_rows = []
for name, r in nd_results.items():
    if r['before'] is not None and r['after'] is not None:
        fairness_rows.append({
            'Dataset':    name,
            'Chronic':    r['chronic'],
            'SPD_Before': r['before']['SPD'],
            'SPD_After':  r['after']['SPD'],
            'DI_Before':  r['before']['DI'],
            'DI_After':   r['after']['DI'],
            'Prevalence_Before': r['before']['Prevalence'],
            'Prevalence_After':  r['after']['Prevalence'],
        })

if fairness_rows:
    fairness_df = pd.DataFrame(fairness_rows)
    print(fairness_df.to_string(index=False))

    # Plot SPD before vs after
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('⚖️ ND Datasets — Fairness Metrics Before/After 7-Algorithm Pipeline',
                 fontsize=13, fontweight='bold')

    # SPD
    spd_data = fairness_df[['Dataset', 'SPD_Before', 'SPD_After']].copy()
    spd_data = spd_data[spd_data['SPD_Before'] != 'N/A']
    if len(spd_data) > 0:
        spd_data['SPD_Before'] = pd.to_numeric(spd_data['SPD_Before'], errors='coerce')
        spd_data['SPD_After']  = pd.to_numeric(spd_data['SPD_After'],  errors='coerce')
        x = range(len(spd_data))
        axes[0].bar([i - 0.2 for i in x], spd_data['SPD_Before'].abs(), 0.35,
                    label='Before', color='#E74C3C', edgecolor='white')
        axes[0].bar([i + 0.2 for i in x], spd_data['SPD_After'].abs(), 0.35,
                    label='After', color='#2ECC71', edgecolor='white')
        axes[0].set_xticks(x)
        axes[0].set_xticklabels(spd_data['Dataset'], rotation=30, ha='right', fontsize=9)
        axes[0].set_title('|SPD| Before vs After', fontweight='bold')
        axes[0].set_ylabel('|Statistical Parity Difference|')
        axes[0].axhline(0.10, ls='--', color='orange', alpha=0.7, label='0.10 threshold')
        axes[0].legend(fontsize=9)

    # DI
    di_data = fairness_df[['Dataset', 'DI_Before', 'DI_After']].copy()
    di_data = di_data[di_data['DI_Before'] != 'N/A']
    if len(di_data) > 0:
        di_data['DI_Before'] = pd.to_numeric(di_data['DI_Before'], errors='coerce')
        di_data['DI_After']  = pd.to_numeric(di_data['DI_After'],  errors='coerce')
        x = range(len(di_data))
        axes[1].bar([i - 0.2 for i in x], di_data['DI_Before'], 0.35,
                    label='Before', color='#E74C3C', edgecolor='white')
        axes[1].bar([i + 0.2 for i in x], di_data['DI_After'], 0.35,
                    label='After', color='#2ECC71', edgecolor='white')
        axes[1].set_xticks(x)
        axes[1].set_xticklabels(di_data['Dataset'], rotation=30, ha='right', fontsize=9)
        axes[1].set_title('Disparate Impact Before vs After', fontweight='bold')
        axes[1].set_ylabel('Disparate Impact')
        axes[1].axhspan(0.80, 1.25, alpha=0.1, color='green', label='80% rule (0.80-1.25)')
        axes[1].legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'plots' / 'nd_fairness_comparison.png', bbox_inches='tight')
    plt.show()
    print('✅ Fairness comparison plot saved')
else:
    print('ℹ️  No fairness data available to plot')

---
## 💾 Stage 7: Export & Final Report

In [ ]:
# ── Save domain-integrated datasets ──
print('💾 Saving domain-integrated datasets...')
domain_dir = OUT_DIR / 'by_domain'
domain_dir.mkdir(exist_ok=True)

for domain, df in domain_dfs.items():
    fname    = f'{domain}_integrated.csv'
    out_path = domain_dir / fname
    df.to_csv(out_path, index=False)
    size = out_path.stat().st_size / 1e3
    print(f'  ✅ {fname:<40} {df.shape} → {size:>8.0f} KB')

# ── Save full integrated dataset ──
print('\n💾 Saving full integrated dataset...')
full_path = OUT_DIR / 'full_integrated_dataset.csv'
full_integrated.to_csv(full_path, index=False)
size_mb = full_path.stat().st_size / 1e6
print(f'  ✅ full_integrated_dataset.csv: {full_integrated.shape} → {size_mb:.1f} MB')

# ── Schema-only summary ──
schema_cols  = [c for c in COMMON_SCHEMA if c in full_integrated.columns]
summary_path = OUT_DIR / 'integration_summary_schema.csv'
full_integrated[schema_cols].to_csv(summary_path, index=False)
print(f'  ✅ integration_summary_schema.csv: {full_integrated[schema_cols].shape}')

# ── Save integration metadata JSON ──
meta = {
    'run_timestamp'      : pd.Timestamp.now().isoformat(),
    'pipeline_version'   : '2.0',
    'total_records'      : int(len(full_integrated)),
    'total_features'     : int(full_integrated.shape[1]),
    'datasets_integrated': list(harmonized.keys()),
    'data_set_datasets'  : list(ds_loaded.keys()),
    'nd_datasets'        : list(nd_loaded.keys()),
    'domains'            : list(domain_dfs.keys()),
    'common_schema'      : COMMON_SCHEMA,
    'per_dataset': {
        name: {
            'records'          : int(len(df)),
            'features'         : int(df.shape[1]),
            'missing_pct'      : float(df.isnull().mean().mean() * 100),
            'has_target'       : bool('target' in df.columns),
            'target_rate'      : float(df['target'].mean()) if 'target' in df.columns and df['target'].notna().any() else None,
            'domain'           : str(df['domain'].iloc[0]) if 'domain' in df.columns else 'unknown',
            'age_availability' : float(df['age'].notna().mean() * 100) if 'age' in df.columns else 0,
            'gender_available' : bool('gender' in df.columns),
            'source_folder'    : 'Data_Set' if name in ds_loaded else 'ND',
        }
        for name, df in harmonized.items()
    },
    'nd_fairness_results': {
        name: {'before': r['before'], 'after': r['after']}
        for name, r in nd_results.items()
        if r['before'] is not None
    }
}

meta_path = OUT_DIR / 'integration_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'  ✅ Metadata JSON saved: {meta_path}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# FINAL REPORT
# ═══════════════════════════════════════════════════════════════════════════

print('=' * 78)
print('  🏥 ND PREPROCESSING & MULTI-DATASET INTEGRATION — FINAL REPORT')
print('=' * 78)
print(f'  Run Timestamp    : {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")}')
print(f'  Output Directory : {OUT_DIR}')
print()

print('─' * 78)
print(f'  {"Dataset":<26} {"Source":<10} {"Domain":<18} {"Records":>9} {"Features":>9}')
print('─' * 78)
for name, df in harmonized.items():
    domain = df['domain'].iloc[0] if 'domain' in df.columns else 'N/A'
    source = 'Data_Set' if name in ds_loaded else 'ND'
    print(f'  {name:<26} {source:<10} {domain:<18} {len(df):>9,} {df.shape[1]:>9}')

print('─' * 78)
print(f'  {"TOTAL INTEGRATED":<26} {"ALL":<10} {"all":<18} {len(full_integrated):>9,} {full_integrated.shape[1]:>9}')
print('=' * 78)
print()

print('  📑 OBJECTIVES ADDRESSED:')
print('  ─' * 26)
print('  Obj 1 (RG1,RG4): Consistent predictions across demographics (all ND datasets)')
print('  Obj 2 (RG1):     Stage 1 pre-processing (7 algorithms) applied to ND datasets')
print('  Obj 3 (RG2):     Each dataset has a clinical risk label as target')
print('  Obj 4 (RG3,RG4): SPD & DI fairness metrics computed before AND after mitigation')
print('  Obj 5 (RG3):     All processing done on local patient data')
print()

print('  📂 Files Generated:')
print('  ' + '-' * 60)
for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file() and f.suffix in ['.csv', '.json', '.png']:
        rel  = f.relative_to(OUT_DIR)
        size = f.stat().st_size / 1e3
        print(f'    {str(rel):<55} {size:>8.0f} KB')
print()
print('  ✅ Pipeline complete! All outputs saved to:', OUT_DIR)